# 🌾 India Districts Crop Production — Full ML Pipeline

**Dataset:** India Districts Crop Production (27,303 rows × 21 columns)  
**Target:** `Production` — crop output in thousand tonnes across Indian districts  
**Years covered:** 2021-22, 2022-23, 2023-24  
**Crops:** 19 types including Rice, Wheat, Maize, Bajra, Pulses, etc.

**Pipeline:**
1. Load & EDA
2. Handle zeros & log-transform skewed features
3. Feature engineering & preprocessing (no leakage)
4. Baseline models (Linear Regression, Decision Tree, Random Forest, SVR)
5. Hyperparameter tuning (GridSearchCV / RandomizedSearchCV)
6. XGBoost & LightGBM
7. Learning curves
8. Residual analysis
9. SHAP (global + local XAI)
10. LIME (local XAI)
11. Final comparison table — all metrics

---
## 1. Imports & Package Setup

We import all required libraries upfront and auto-install any ML/XAI packages that may not already be present in the environment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import subprocess, sys
for pkg in ['xgboost', 'lightgbm', 'shap', 'lime']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=False)

print('✅ All packages ready.')

---
## 2. Load & Explore the Data

### About this dataset
Each row represents a **crop grown in a specific district in a specific year**. Features include:
- **Geographic:** State, District, Latitude, Longitude
- **Agronomic:** Crop type, Year
- **Soil:** Nitrogen, Phosphorus, Potassium (NPK), Organic Carbon, Soil pH
- **Weather:** Temperature, Humidity, Precipitation, Sunshine hours
- **Pre-encoded:** State_Encoded, District_Encoded, Crop_Encoded (label-encoded versions of categoricals)
- **Target:** `Production` (in thousand tonnes)

Note: `Country` is always 'India' — we drop it. `Ind_adm2_ID` is just a geographic ID — not a useful predictor.

In [ ]:
df = pd.read_csv('India_Districts_Crop_Production_Processed.csv')

print('Shape:', df.shape)
print('\nColumns:', list(df.columns))
df.head()

In [ ]:
# Statistical summary — check min/max/mean for anomalies
# Key things to notice:
# - Production min = 0, 25th percentile = 0 → more than half the rows are zero production!
# - Production max = 3090 but mean = 90 → extreme right skew
df.describe()

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print('\n✅ No missing values in this dataset — no imputation needed.')

In [ ]:
# Dataset composition
print('Unique States :', df['State'].nunique())
print('Unique Districts:', df['District'].nunique())
print('Unique Crops  :', df['Crop'].nunique(), '→', list(df['Crop'].unique()))
print('Years covered :', list(df['Year'].unique()))

print(f'\nZero-production rows: {(df["Production"]==0).sum()} out of {len(df)} ({(df["Production"]==0).mean()*100:.1f}%)')
print('This means many districts did not grow a particular crop in a given year.')

### Visualising the Data

In [ ]:
# Top 10 crops by total production
top_crops = df.groupby('Crop')['Production'].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(10, 5))
top_crops.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Top 10 Crops by Total Production (All Years)')
plt.ylabel('Total Production (thousand tonnes)')
plt.xlabel('Crop')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 states by production
top_states = df.groupby('State')['Production'].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(10, 5))
top_states.plot(kind='bar', color='tomato', edgecolor='white')
plt.title('Top 10 States by Total Production')
plt.ylabel('Total Production (thousand tonnes)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Year-over-year production trend
yearly = df.groupby('Year')['Production'].sum()
plt.figure(figsize=(7, 4))
yearly.plot(kind='bar', color=['#5cb85c', '#337ab7', '#f0ad4e'], edgecolor='white')
plt.title('Total Production by Year')
plt.ylabel('Total Production (thousand tonnes)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap — numeric features only
# Drop identifier columns before computing correlations
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
plt.figure(figsize=(12, 9))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix — All Numeric Features\n(Look for features highly correlated with Production)')
plt.tight_layout()
plt.show()

---
## 3. 📐 Handling Zeros & Log-Transform

### The zero-production challenge
Over 60% of rows have `Production = 0`. This happens when a district simply didn't grow that crop that year. These are **valid data points** — they shouldn't be removed. However, they do affect the distribution heavily.

### Why log-transform?
`Production` is extremely right-skewed (skewness ≈ 4.3) — a few districts produce thousands of tonnes while most produce very little. Without transformation:
- Linear models try to fit massive outliers and ignore small values
- Error metrics are dominated by large-production districts
- Tree splits get biased towards outlier thresholds

We use `log1p(x) = log(x + 1)` which safely handles zeros (`log(0+1) = 0`) and compresses the range dramatically.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['Production'], bins=60, color='tomato', edgecolor='white')
axes[0].set_title(f'Production — Original\nSkewness: {df["Production"].skew():.2f}')
axes[0].set_xlabel('Production (thousand tonnes)')

log_prod = np.log1p(df['Production'])
axes[1].hist(log_prod, bins=60, color='steelblue', edgecolor='white')
axes[1].set_title(f'Production — After log1p\nSkewness: {log_prod.skew():.2f}')
axes[1].set_xlabel('log1p(Production)')

plt.suptitle('log1p transforms extreme skew into a workable distribution', fontsize=12)
plt.tight_layout()
plt.show()

print(f'Production skewness BEFORE: {df["Production"].skew():.2f}')
print(f'Production skewness AFTER : {log_prod.skew():.2f}')